# Hagerty — Acquisition Channel Analysis (starter)

This notebook is just here to get you past setup — it loads the three tables and shows a quick peek so you can start thinking about the problem instead of about file paths. **Everything past the peek is yours.** There's no prescribed path, and no need to keep or use these cells as written; rework them however you like.

**Where we're starting:** *which acquisition channels bring in the customers most likely to become paying customers?*


### Imports

In [11]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# -----------------------------
# Add project root to sys.path for module imports
# -----------------------------
root = "/Users/phillipsmith/Desktop/Python/customer_conversion_proj"
sys.path.append(root)

# -----------------------------
# Import external python modules
# -----------------------------
from pathlib import Path

### Config & Data Loading

In [ ]:
# -----------------------------
# Display options
# -----------------------------
pd.set_option('display.max_rows', None) # reset_option to compact dataframe view
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.6f}'.format)

# -----------------------------
# Raw data location (immutable source files)
# -----------------------------
raw_files = Path(root) / "data" / "raw"

# -----------------------------
# Load every CSV / Excel file in data/raw/ into a dict keyed by file name.
# Format-agnostic so it works whether the files are .csv, .xlsx, or .xls.
# -----------------------------
def load_file(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    return pd.read_excel(path)  # .xlsx / .xls

datasets = {
    f.stem: load_file(f)
    for f in sorted(raw_files.glob("*"))
    if f.suffix.lower() in {".csv", ".xlsx", ".xls"}
}

if not datasets:
    print(f"No CSV/Excel files found in {raw_files}")

# Initialize datasets; copy to prevent mutation
customers = datasets["customers"].copy()
transactions = datasets["transactions"].copy()
attributes = datasets["customer_attributes"].copy()

# Parse the date columns
customers["account_created_date"] = pd.to_datetime(customers["account_created_date"])
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

In [10]:
# print name, shape, and columns of dataset(s)
for name, df in datasets.items():
    print(name)
    print(f"{df.shape[0]} rows x {df.shape[1]} cols")
    print(df.columns)
    print()

customer_attributes
1112 rows x 5 cols
Index(['customer_id', 'vehicle_count', 'top_vehicle_value', 'vehicle_category',
       'age_band'],
      dtype='object')

customers
1145 rows x 5 cols
Index(['customer_id', 'account_created_date', 'acquisition_channel',
       'signup_product', 'state'],
      dtype='object')

transactions
648 rows x 5 cols
Index(['customer_id', 'transaction_date', 'product_line', 'transaction_type',
       'amount'],
      dtype='object')



---
## Your turn


### Problem Framing

**Define Value:** Profitability within 6 months after a transaction data stream
    - Understand transactions trends amongst each product line

**Business Decision:** Optimize investment allocation

Check refunds based on product lines
    - partial refunds?
    - allows refunds?

Transactions dates are point-in-time-correct as of today
    - retention period for allowed refunds
    - Marketplace
        - 7% of full vehicle price and refunds (- sign)
    - Other PL
        - 100% of full vehicle price for payments and refunds

Features
    - customer tenure

Notes
    - business rules drive contextual scenarios
    - Segmentation to keep outliers in mind and compare customers fairly

In [ ]:
customers.head()

# find dupes count
print(customers["customer_id"].duplicated().sum())

# see dupes
print(customers[customers["customer_id"].duplicated()])

# drop dupes
customers = customers.drop_duplicates(subset=["customer_id"], keep="first")



0
Empty DataFrame
Columns: [customer_id, account_created_date, acquisition_channel, signup_product, state]
Index: []


,customer_id,account_created_date,acquisition_channel,signup_product,state
0,100219,2023-12-14,organic,insurance,PA
1,100810,2023-12-10,social,marketplace,WA
2,100502,2024-01-14,paid_search,insurance,MI
3,100650,2024-02-07,paid_search,events,MI
4,100324,2024-01-26,organic,insurance,FL


In [28]:
transactions.head()

,customer_id,transaction_date,product_line,transaction_type,amount
0,101125,2024-02-16,insurance,payment,868.690000
1,100353,2024-10-05,hdc_membership,payment,75.000000
2,101060,2025-02-11,insurance,payment,1635.140000
3,100751,2023-09-09,marketplace,payment,17390.390000
4,100108,2024-04-18,insurance,payment,2170.370000


In [29]:
attributes.head()

# print(attributes["customer_id"].duplicated().sum())

,customer_id,vehicle_count,top_vehicle_value,vehicle_category,age_band
0,100451,1.000000,46666.000000,muscle,45_59
1,100556,NaN,NaN,NaN,under_30
2,100058,1.000000,9368.000000,muscle,60_plus
3,100716,1.000000,44567.000000,classic,45_59
4,100743,NaN,NaN,NaN,under_30


In [36]:
# merge datasets

customer_merged = (
    customers
    .merge(transactions, on="customer_id", how="left")
    .merge(attributes, on="customer_id", how="left")
)

print(customer_merged.shape)
print(customer_merged["customer_id"].duplicated().sum())

customer_merged[customer_merged["customer_id"].duplicated()].head(20)

(1471, 13)
329


,customer_id,account_created_date,acquisition_channel,signup_product,state,transaction_date,product_line,transaction_type,amount,vehicle_count,top_vehicle_value,vehicle_category,age_band
8,100108,2023-11-09,organic,insurance,NY,2024-09-01,marketplace,payment,50890.340000,2.000000,69067.000000,exotic,60_plus
9,100108,2023-11-09,organic,insurance,NY,2024-09-14,marketplace,refund,27524.450000,2.000000,69067.000000,exotic,60_plus
10,100108,2023-11-09,organic,insurance,NY,2024-05-25,insurance,refund,774.810000,2.000000,69067.000000,exotic,60_plus
11,100108,2023-11-09,organic,insurance,NY,2024-05-25,marketplace,refund,31063.200000,2.000000,69067.000000,exotic,60_plus
12,100108,2023-11-09,organic,insurance,NY,2024-04-18,marketplace,payment,31063.200000,2.000000,69067.000000,exotic,60_plus
13,100108,2023-11-09,organic,insurance,NY,2024-04-18,hdc_membership,payment,75.000000,2.000000,69067.000000,exotic,60_plus
25,100210,2024-02-21,organic,marketplace,FL,2024-08-16,insurance,payment,745.700000,NaN,NaN,NaN,under_30
36,100391,2023-11-13,organic,membership,CA,2024-10-23,events,payment,75.000000,2.000000,40975.000000,NaN,30_44
37,100391,2023-11-13,organic,membership,CA,2024-10-23,insurance,payment,1765.830000,2.000000,40975.000000,NaN,30_44
53,100024,2023-10-28,organic,insurance,CA,2023-11-05,events,payment,250.000000,NaN,NaN,NaN,under_30
